# Klebsiella metadata

In [1]:
import pandas as pd
#paths

# No additional imports are needed; you already imported 'pandas as pd' above.
data_dir = "/home/dca36/rds/rds-floto-bacterial-4k08a2yyQLw/david"
metadata_file = data_dir + "/final/metadata_final_curated_slimmed.tsv"

metadata = pd.read_csv(metadata_file, sep="\t", low_memory=False)
display(metadata.head())

,Sample,is_kpsc,kpsc_final_list,is_refseq,is_nctc,is_mgh78578,is_complete_norway_genome,have_transcriptome,species,species_match,...,collection_date,collection_date_parsed,year_parsed,dev_stage,scientific_name,center_name,amr_study,study_setting,assembly_file,gff_file
0,SAMD00002693,False,False,False,False,False,False,False,Klebsiella ornithinolytica,strong,...,NaN,NaN,NaN,NaN,Raoultella ornithinolytica,GIFU_MED,NaN,NaN,NaN,david/raw/klebsiella_gff3/SAMD00002693.bakta.g...
1,SAMD00008836,False,False,False,False,False,False,False,Klebsiella terrigena,strong,...,NaN,NaN,NaN,NaN,Klebsiella pneumoniae subsp. ozaenae,GIFU_MED,NaN,NaN,NaN,david/raw/klebsiella_gff3/SAMD00008836.bakta.g...
2,SAMD00008944,False,False,False,False,False,False,False,Klebsiella oxytoca,strong,...,NaN,NaN,NaN,NaN,Klebsiella oxytoca,GIFU_MED,NaN,NaN,NaN,david/raw/klebsiella_gff3/SAMD00008944.bakta.g...
3,SAMD00009480,False,False,False,False,False,False,False,Klebsiella aerogenes,strong,...,NaN,NaN,NaN,NaN,Klebsiella aerogenes,GIFU_MED,NaN,NaN,NaN,david/raw/klebsiella_gff3/SAMD00009480.bakta.g...
4,SAMD00049595,True,True,False,False,False,False,False,Klebsiella pneumoniae,strong,...,2009-01-01,2009/01/01,2009.0,NaN,Klebsiella pneumoniae,KYOTO_GM,NaN,NaN,seb/assemblies_2/klebsiella_pneumoniae__01/SAM...,david/raw/klebsiella_gff3/SAMD00049595.bakta.g...


In [2]:
# Select kpsc_final_list True
kpsc_final_metadata = metadata[metadata['kpsc_final_list']]
print(f"Number of samples in kpsc_final_list: {len(kpsc_final_metadata)}")

# # then randomly sample 2500 from ST258
# st258_metadata = kpsc_final_metadata[kpsc_final_metadata['Sublineage'] == 'SL258']
# print(f"Number of samples in ST258: {len(st258_metadata)}")
# # Randomly sample 2500 from ST258
# st258_sampled_metadata = st258_metadata.sample(n=2500)
# print(f"Number of samples in ST258 sampled: {len(st258_sampled_metadata)}")

# # Save the sampled metadata to david/processed/panaroo_run/st258_sampled.tsv
# st258_sampled_metadata.to_csv(f"/home/dca36/rds/rds-floto-bacterial-4k08a2yyQLw/david/processed/panaroo_run/SL258_sampled.tsv", sep="\t")



Number of samples in kpsc_final_list: 79450


In [3]:
# Check how many kpsc_final_metadata have entries for assembly_file and gff_file
print(f"Number of kpsc_final with assembly_file: {kpsc_final_metadata['assembly_file'].notna().sum()}")
print(f"Number of kpsc_final_metadata with gff_file: {kpsc_final_metadata['gff_file'].notna().sum()}")

Number of kpsc_final with assembly_file: 79450
Number of kpsc_final_metadata with gff_file: 79450


In [4]:
# Double check that all with sublineage not null are in kpsc_final_list
sl_samples = kpsc_final_metadata[kpsc_final_metadata['Sublineage'].notna()]
# Number of samples with sublineage not null
print(f"Number of samples with sublineage not null: {len(sl_samples)}")
# Number of samples with sublineage not null and in kpsc_final_list
sl_samples = sl_samples[sl_samples['kpsc_final_list']]
print(f"Number of samples with sublineage not null and in kpsc_final_list: {len(sl_samples)}")

Number of samples with sublineage not null: 79042
Number of samples with sublineage not null and in kpsc_final_list: 79042


In [5]:
# Filter to those samples in kpsc_final_list True
kpsc_final_list_samples = metadata[metadata['kpsc_final_list']]
print(f"Number of samples in kpsc_final_list: {len(kpsc_final_list_samples)}")

Number of samples in kpsc_final_list: 79450


In [ ]:
# Check we have one is_mgh78578 sample
is_mgh78578_samples = kpsc_final_list_samples[kpsc_final_list_samples['is_mgh78578']]
print(f"Number of samples with is_mgh78578: {len(is_mgh78578_samples)}")
# Print the sublineage and clonal group for the one sample
print(is_mgh78578_samples[['Sublineage', 'Clonal group']])


Number of samples with is_mgh78578: 1
      Sublineage Clonal group
86654       SL38         CG38


In [ ]:
# Ranking sublineages by size, choose all those with more than 100 samples
# For each, count how many are is_refseq True and how many is_complete_norway_genome True

strain_type_column = "Sublineage"
min_samples = 250

# Get value counts for each sublineage
strain_type_sizes = kpsc_final_list_samples[strain_type_column].value_counts()
filtered_strain_types = strain_type_sizes[strain_type_sizes > min_samples].index.tolist()

# For each of the largest sublineages, count the total and how many are is_refseq True
results = []
for sublineage in filtered_strain_types:
    total = strain_type_sizes[sublineage]
    n_is_refseq = kpsc_final_list_samples[
        (kpsc_final_list_samples[strain_type_column] == sublineage) & 
        (kpsc_final_list_samples['is_refseq'])
    ].shape[0]
    n_is_complete_norway_genome = kpsc_final_list_samples[
        (kpsc_final_list_samples[strain_type_column] == sublineage) & 
        (kpsc_final_list_samples['is_complete_norway_genome'])
    ].shape[0]
    results.append({
        'Sublineage': sublineage, 
        'count': total, 
        'count_is_refseq': n_is_refseq, 
        'count_is_complete_norway_genome': n_is_complete_norway_genome
    })

sl_df = pd.DataFrame(results)
# Print the total number of samples in the sublineages with > min_samples samples
print(f"Total number of samples in the sublineages with > {min_samples} samples: {sl_df['count'].sum()}")
print(f"The total number of complete norway genomes in the sublineages with > {min_samples} samples: {sl_df['count_is_complete_norway_genome'].sum()}")
# print whole dataframe
display(sl_df)

Total number of samples in the sublineages with > 250 samples: 56406
The total number of complete norway genomes in the sublineages with > 250 samples: 199


,Sublineage,count,count_is_refseq,count_is_complete_norway_genome
0,SL258,16229,947,10
1,SL147,5096,177,3
2,SL17,4630,113,21
3,SL307,4449,159,8
4,SL15,3647,161,7
5,SL14,2521,72,6
6,SL37,1871,102,17
7,SL45,1803,52,13
8,SL101,1610,73,3
9,SL231,1049,57,3


In [8]:
# Clonal groups in a given sublineage
sublineage = "SL307"
clonal_groups = kpsc_final_list_samples[kpsc_final_list_samples['Sublineage'] == sublineage]['Clonal group'].unique()
# Count how many samples are in each clonal group
clonal_group_counts = kpsc_final_list_samples[kpsc_final_list_samples['Clonal group'].isin(clonal_groups)]['Clonal group'].value_counts()
# Print in descending order
print(clonal_group_counts.sort_values(ascending=False))


Clonal group
CG307      4437
CG10814      10
CG13774       1
CG13830       1
Name: count, dtype: int64


In [9]:
def run_panaroo_on_small_sublineages(n: int):
    # Calculate the total number of samples in sublineages of size < n
    small_sublineages = sublineage_counts[sublineage_counts < n]
    # Filter to metadata file to only include the SLs of size < n
    small_sublineages_metadata = kpsc_final_list_samples[kpsc_final_list_samples['Sublineage'].isin(small_sublineages.index)]
    print(f"Total number of samples in sublineages of size < {n}: {len(small_sublineages_metadata)}")
    print(f"Total number of sublineages of size < {n}: {len(small_sublineages)}")

    # Save the metadata file to david/processed/panaroo_run/sublineages_size_<n>.tsv
    small_sublineages_metadata.to_csv(f"/home/dca36/rds/rds-floto-bacterial-4k08a2yyQLw/david/processed/panaroo_run/sublineages_smaller_than_{n}.tsv", sep="\t")
    print(f"Saved metadata file to david/processed/panaroo_run/sublineages_smaller_than_{n}.tsv")
    return small_sublineages_metadata

In [10]:
# Define sublineage_other function
final_kpsc_samples = metadata[metadata['kpsc_final_list']]
print(f"Number of samples in final kpsc set: {len(final_kpsc_samples)}")

# It takes all metadata samples that are in final set, subtracts all samples in a list of Clonal groups provided
sublineage = "SL17"
# How many samples are in the sublineage
sublineage_samples = metadata[metadata['Sublineage'] == sublineage]
print(f"Number of samples in {sublineage}: {len(sublineage_samples)}")
# Subtract clonal groups from the sublineage
subtract_clonal_groups = ["CG15"]
sublineage_other_metadata = sublineage_samples[~sublineage_samples['Clonal group'].isin(subtract_clonal_groups)]
print(f"Number of samples in {sublineage} other: {len(sublineage_other_metadata)}")
# Save the metadata to david/processed/panaroo_run/{sublineage}_other.tsv
sublineage_other_metadata.to_csv(f"/home/dca36/rds/rds-floto-bacterial-4k08a2yyQLw/david/processed/panaroo_run/{sublineage}_other.tsv", sep="\t")

Number of samples in final kpsc set: 79450
Number of samples in SL17: 4702
Number of samples in SL17 other: 4702


In [11]:
# Pack sublineages (each < 500 samples) into greedy batches of >= 1500 total samples.
# We build batches from `final_kpsc_samples` only, and always save the final remainder batch.

out_dir = "/home/dca36/rds/rds-floto-bacterial-4k08a2yyQLw/david/processed/panaroo_run"
max_sublineage_size = 500

# Sublineage sample counts (within the final kpsc set)
sl_sample_counts = final_kpsc_samples["Sublineage"].value_counts().sort_values(ascending=False)

# Only keep SLs below the max size
sl_sample_counts_less_than_500 = sl_sample_counts[sl_sample_counts < max_sublineage_size]

# Metadata for the SLs below 500
sl_less_than_500_metadata = final_kpsc_samples[final_kpsc_samples['Sublineage'].isin(sl_sample_counts_less_than_500.index)]
# Look as 'species' counts for the SLs below 500
sl_less_than_500_species_counts = sl_less_than_500_metadata['species'].value_counts()
# Print the species counts for the SLs below 500
print(sl_less_than_500_species_counts)

species
Klebsiella pneumoniae                                 20410
Klebsiella variicola subsp. variicola                  2666
Klebsiella quasipneumoniae subsp. similipneumoniae     2489
Klebsiella quasipneumoniae subsp. quasipneumoniae      1080
Klebsiella quasivariicola                                75
Klebsiella africana                                      17
Klebsiella variicola subsp. tropica                      14
Name: count, dtype: int64


In [12]:
# Slice out metadata for species == "Klebsiella pneumoniae"
klebsiella_pneumoniae_metadata = sl_less_than_500_metadata[sl_less_than_500_metadata['species'] == "Klebsiella pneumoniae"]
print(f"Number of samples in Klebsiella pneumoniae: {len(klebsiella_pneumoniae_metadata)}")

# Slice metadata for each species in sl_less_than_500_metadata, save each to david/processed/panaroo_run/species_<species>.tsv
for species in sl_less_than_500_species_counts.index:
    # Skip Klebsiella pneumoniae - will batch this separately, by Sublineage
    if species != "Klebsiella pneumoniae":
        species_metadata = sl_less_than_500_metadata[sl_less_than_500_metadata['species'] == species]
        print(f"Number of samples in {species}: {len(species_metadata)}")
        species_metadata.to_csv(f"/home/dca36/rds/rds-floto-bacterial-4k08a2yyQLw/david/processed/panaroo_run/species_{species}.tsv", sep="\t")
        print(f"Saved metadata for {species} to david/processed/panaroo_run/species_{species}.tsv")

Number of samples in Klebsiella pneumoniae: 20410
Number of samples in Klebsiella variicola subsp. variicola: 2666
Saved metadata for Klebsiella variicola subsp. variicola to david/processed/panaroo_run/species_Klebsiella variicola subsp. variicola.tsv
Number of samples in Klebsiella quasipneumoniae subsp. similipneumoniae: 2489
Saved metadata for Klebsiella quasipneumoniae subsp. similipneumoniae to david/processed/panaroo_run/species_Klebsiella quasipneumoniae subsp. similipneumoniae.tsv
Number of samples in Klebsiella quasipneumoniae subsp. quasipneumoniae: 1080
Saved metadata for Klebsiella quasipneumoniae subsp. quasipneumoniae to david/processed/panaroo_run/species_Klebsiella quasipneumoniae subsp. quasipneumoniae.tsv
Number of samples in Klebsiella quasivariicola: 75
Saved metadata for Klebsiella quasivariicola to david/processed/panaroo_run/species_Klebsiella quasivariicola.tsv
Number of samples in Klebsiella africana: 17
Saved metadata for Klebsiella africana to david/processe

In [13]:
# For Klebsiella pneumoniae, batch by sublineage

sl_list = klebsiella_pneumoniae_metadata['Sublineage'].unique().tolist()
sl_sizes = klebsiella_pneumoniae_metadata['Sublineage'].value_counts()
max_sublineage_size = 500
print(f"Found {len(sl_list)} sublineages with < {max_sublineage_size} samples.")
# Print top 10 sublineages
print(sl_sizes.head(10))

target_total = 1000
batch_i = 0
current_sls = []
current_parts = []
current_count = 0

for sl in sl_list:
    sl_df = final_kpsc_samples.loc[final_kpsc_samples["Sublineage"] == sl]

    current_sls.append(sl)
    current_parts.append(sl_df)
    current_count += len(sl_df)

    # Save when we reach/exceed the target (including the SL that pushes it over)
    if current_count >= target_total:
        batch_df = pd.concat(current_parts, ignore_index=True)
        out_path = f"{out_dir}/kp_rare_sublineage_batch_{batch_i}.tsv"
        batch_df.to_csv(out_path, sep="\t")

        print(f"Saved batch {batch_i}: {current_count} samples across {len(current_sls)} sublineages -> {out_path}")

        batch_i += 1
        current_sls = []
        current_parts = []
        current_count = 0

# Save final remainder batch (often < 1500)
if current_parts:
    batch_df = pd.concat(current_parts, ignore_index=True)
    out_path = f"{out_dir}/next_sublineage_batch_{batch_i}.tsv"
    batch_df.to_csv(out_path, sep="\t")
    print(f"Saved final partial batch {batch_i}: {len(batch_df)} samples across {len(current_sls)} sublineages -> {out_path}")


Found 1479 sublineages with < 500 samples.
Sublineage
SL405     499
SL152     422
SL661     409
SL323     398
SL252     393
SL86      391
SL2004    380
SL1       320
SL76      317
SL383     296
Name: count, dtype: int64
Saved batch 0: 1133 samples across 10 sublineages -> /home/dca36/rds/rds-floto-bacterial-4k08a2yyQLw/david/processed/panaroo_run/kp_rare_sublineage_batch_0.tsv
Saved batch 1: 1133 samples across 9 sublineages -> /home/dca36/rds/rds-floto-bacterial-4k08a2yyQLw/david/processed/panaroo_run/kp_rare_sublineage_batch_1.tsv
Saved batch 2: 1002 samples across 7 sublineages -> /home/dca36/rds/rds-floto-bacterial-4k08a2yyQLw/david/processed/panaroo_run/kp_rare_sublineage_batch_2.tsv
Saved batch 3: 1020 samples across 13 sublineages -> /home/dca36/rds/rds-floto-bacterial-4k08a2yyQLw/david/processed/panaroo_run/kp_rare_sublineage_batch_3.tsv
Saved batch 4: 1236 samples across 10 sublineages -> /home/dca36/rds/rds-floto-bacterial-4k08a2yyQLw/david/processed/panaroo_run/kp_rare_subli